<a href="https://colab.research.google.com/github/kazarimm/triage-copilot/blob/main/notebooks/Staff_Triage_ML_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Triage Copilot Model**
by Marcos Salazar



**Purpose:**

This notebook trains a machine learning pipeline (TF-IDF + LinearSVC) to classify bilingual student inquiries into 6 administrative categories (Admissions, Advising, Financial Services, ESL, Navigate, Appointment).

**Expected Outcomes:**

The trained Scikit-learn pipeline for the FastAPI backend. (model.joblib)

AND

The cleaned reference dataset for the learning feedback loop. (training_data.csv)

Installing Scikit-learn.

In [9]:
!pip install -U scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 69.7 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


Importing the library and verifying library version.

In [19]:
import sklearn

print(sklearn.__version__)

1.9.0


**Setup:**

Importing necessary libraries.

In [20]:
#Data Manipulation
import pandas as pd
import numpy as np
import random

#Visualization
import matplotlib.pyplot as plt
import seaborn as sns

#Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

#Export tools
import joblib

#Reproducibility
random.seed(42)
np.random.seed(42)

Loading data from (training_phrases.csv)

In [12]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [21]:
phrases_df = pd.read_csv('/content/drive/MyDrive/Triage-Copilot-Folder/training_phrases.csv')
print(phrases_df.shape)
phrases_df.head()

(100, 4)


,Category,Subcategory,Language,Phrase
0,Admissions / Enrollment,Application Assistance,EN,I need help with my application
1,Admissions / Enrollment,Application Assistance,EN,Can someone help me finish applying
2,Admissions / Enrollment,Application Assistance,EN,I'm stuck on the admissions application
3,Admissions / Enrollment,Application Assistance,ES,Necesito ayuda con mi aplicación
4,Admissions / Enrollment,Application Assistance,ES,¿Alguien me puede ayudar a terminar de aplicar?


Successfully uploaded training data.

Using the same training data, I will generate more training data from the loaded phrases by creating minor variations.

In [22]:
# Function that generates variants of the pre-existing data

def augment_phrase(phrase):
  variants = [phrase, phrase.lower(), phrase + "?", phrase + "."]
  return random.choice(variants)

#Generative goal of 60 examples per department (combat overfitting)
TARGET_PER_CATEGORY = 60

rows = []

for category in phrases_df["Category"].unique():
  category_rows = phrases_df[phrases_df["Category"] == category]

  #Counter - Artificial samples created for current category.
  generated = 0

  while generated < TARGET_PER_CATEGORY:

    sample = category_rows.sample(1).iloc[0]
    phrase = augment_phrase(sample["Phrase"])
    rows.append({
        "text": phrase,
        "category": sample["Category"],
        "subcategory": sample["Subcategory"]
    })
    generated += 1

#Create new dataframe for Artificial samples/training data.
newData = pd.DataFrame(rows).drop_duplicates(subset="text").reset_index(drop=True)
newData.to_csv("training_data.csv", index=False)

Let's analyze the new data.

In [23]:
print(newData.shape)
newData.head(10)

(229, 3)


,text,category,subcategory
0,I need a verification letter for enrollment,Admissions / Enrollment,Verification Letter
1,I need help with my application,Admissions / Enrollment,Application Assistance
2,¿Alguien me puede ayudar a terminar de aplicar??,Admissions / Enrollment,Application Assistance
3,necesito ayuda con mi aplicación,Admissions / Enrollment,Application Assistance
4,quiero inscribirme en un curso sin crédito,Admissions / Enrollment,Non-credit Registration
5,how do i find my student id number,Admissions / Enrollment,I need an N#
6,Necesito mi número N,Admissions / Enrollment,I need an N#
7,I need to finish my enrollment form,Admissions / Enrollment,Enrollment Form Completion
8,How do I find my student ID number.,Admissions / Enrollment,I need an N#
9,I don't have my N number,Admissions / Enrollment,I need an N#
